# Statistical Analysis

## Hypothesis Testing

### Objectives

* Carry out Hypotheses listed within README.md utilising statistical analysis, paired with visuals in order to draw conclusions and thereby prove or disprove

### Inputs

* Cleaned data prepared in ETL

### Output

* Responses to hypotheses

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'd:\\Coding Cooler Dump\\Credit Card Customers Analysis\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'd:\\Coding Cooler Dump\\Credit Card Customers Analysis'

# Importing Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

---

Loading the Dataset

In [5]:
df = pd.read_csv("Dataset/CleanedData/cleanData.csv")
df.head()

,winner,t1_champ1id,t1_champ2id,t1_champ3id,t1_champ4id,t1_champ5id,t2_champ1id,t2_champ2id,t2_champ3id,t2_champ4id,...,t1_champ1id_pickrate,t1_champ2id_pickrate,t1_champ3id_pickrate,t1_champ4id_pickrate,t1_champ5id_pickrate,t2_champ1id_pickrate,t2_champ2id_pickrate,t2_champ3id_pickrate,t2_champ4id_pickrate,t2_champ5id_pickrate
0,1,8,432,96,11,112,104,498,122,238,...,0.62,0.72,0.63,1.15,0.34,0.28,1.52,0.76,1.25,2.52
1,1,119,39,76,10,35,54,25,120,157,...,0.99,0.44,0.33,0.39,0.50,0.49,1.15,0.39,1.53,1.18
2,1,18,141,267,68,38,69,412,126,24,...,2.52,1.91,0.91,0.31,0.59,0.54,2.52,0.62,1.29,1.05
3,1,57,63,29,61,36,90,19,412,92,...,0.84,0.75,1.72,1.27,0.30,0.73,1.14,2.52,1.18,1.05
4,1,19,29,40,119,134,37,59,141,38,...,1.14,1.72,1.69,0.99,0.73,1.05,1.17,1.91,0.59,1.32


---

# Hypothesis 1: Using Levene’s Test for Equality of Variances

To test if there is a correlation between pick and win rates.

* Alternative Hypothesis: Champion win rates exhibit a significant negative correlation with their pick rates, where lower popularity scores are associated with higher win rate variance.

* Null Hypothesis: There is no relationship between a champion's pick rate and win rate variance.

For this section of code, I employed the aid of Co-Pilot, in order to swiftly find the top and bottom 30% of champions based on pick rate. I confirmed that the code functioned correctly, and compared values to data I had available, before continuing to use it, as I had encountered issues regarding pick rate in the ETL section.

In [6]:
champion_columns = [
    f"{team}_champ{champion_number}id"
    for team in ("t1", "t2")
    for champion_number in range(1, 6)
]

champion_picks = df[champion_columns].stack().value_counts()
champion_pick_rates = (champion_picks / champion_picks.sum() * 100).rename("pick_rate")
champion_pick_rates.index.name = "champion_id"
champion_pick_rates = champion_pick_rates.sort_values(ascending=False)

group_size = int(np.ceil(len(champion_pick_rates) * 0.30))
top_30_percent = champion_pick_rates.head(group_size).reset_index()
bottom_30_percent = champion_pick_rates.tail(group_size).sort_values().reset_index()

top_30_percent["group"] = "Top 30%"
bottom_30_percent["group"] = "Bottom 30%"
champion_groups = pd.concat(
    [top_30_percent, bottom_30_percent],
    ignore_index=True
)[["champion_id", "pick_rate", "group"]]

print(champion_groups.groupby("group").size())
champion_groups

group
Bottom 30%    42
Top 30%       42
dtype: int64


,champion_id,pick_rate,group
0,18,2.523848,Top 30%
1,412,2.521889,Top 30%
2,67,2.068635,Top 30%
3,141,1.911935,Top 30%
4,64,1.785595,Top 30%
...,...,...,...
79,120,0.387441,Bottom 30%
80,163,0.388028,Bottom 30%
81,10,0.388420,Bottom 30%
82,14,0.408007,Bottom 30%


With these two groups created, I am now able to conduct the statistical test. Initially, I conducted this statistical analysis on my own, but found myself struggling, so eventually turned once again to co-pilot, aiding in the recording of win rates to be used for the analysis.

In [7]:
from scipy import stats

win_rate = [f"{column}_winrate" for column in champion_columns]
champion_win_rate_data = pd.concat(
    [
        df[[champion_column, win_rate_column]].rename(
            columns={champion_column: "champion_id", win_rate_column: "win_rate"}
        )
        for champion_column, win_rate_column in zip(champion_columns, win_rate)
    ],
    ignore_index=True
)

champion_win_rates = champion_win_rate_data.groupby("champion_id")["win_rate"].mean()
top_win_rates = champion_win_rates.loc[top_30_percent["champion_id"]]
bottom_win_rates = champion_win_rates.loc[bottom_30_percent["champion_id"]]

stat, p_value = stats.levene(top_win_rates, bottom_win_rates)
alpha = 0.05

print(f"Test statistic: {stat}, p-value: {p_value}")

if p_value < alpha:
    print("Reject the null hypothesis: The variances are significantly different.")
else:
    print("Fail to reject the null hypothesis: The variances are not significantly different.")

Test statistic: 0.10608927084952927, p-value: 0.7454708538456272
Fail to reject the null hypothesis: The variances are not significantly different.


Examinining the p_value, it is clear that it is greater than the alpha value, and therefore this hypothesis is not statistically strong enough to be proven. The test statistic being so close to 0 suggests that the opposite of my hypothesis is true instead, and characters with low pick rate have almost equal volatility to those in high pick rate.

One element to consider is the the % breaks I used for this analysis. Perhaps it would have net a different result had i only used to top and bottom 20% rather than 30%

---

# Hypothesis 2: Using Chi-Square Test

To test the average win rate.

* Alternative Hypothesis: Individual champion win rates significantly deviate from a balanced 50% baseline distribution, indicating systemic character imbalance in the current meta.

* Null Hypothesis: Every champion possesses an identical, perfectly balanced win rate of exactly 50%.

I once again employed the aid of Co-Pilot, as I was unsure how to tackle a Chi Square test which functionally was run for each champion. From this, I learned of the bonferonni correction.

In [12]:
from scipy import stats

# Count wins and losses for every champion appearance.
champion_columns = [
    f"{team}_champ{champion_number}id"
    for team in ("t1", "t2")
    for champion_number in range(1, 6)
]

champion_records = []
for champion_column in champion_columns:
    team = 1 if champion_column.startswith("t1_") else 2
    champion_records.append(
        pd.DataFrame({
            "champion_id": df[champion_column],
            "win": (df["winner"] == team).astype(int)
        })
    )

champion_results = pd.concat(champion_records, ignore_index=True)
champion_summary = champion_results.groupby("champion_id")["win"].agg(
    appearances="count",
    wins="sum"
)
champion_summary["losses"] = champion_summary["appearances"] - champion_summary["wins"]
champion_summary["win_rate"] = champion_summary["wins"] / champion_summary["appearances"] * 100

# Test each champion's observed wins/losses against an expected 50/50 split.
alpha = 0.05
chi_square_results = []
for champion_id, row in champion_summary.iterrows():
    observed = [row["wins"], row["losses"]]
    expected = [row["appearances"] / 2, row["appearances"] / 2]
    statistic, p_value = stats.chisquare(observed, f_exp=expected)
    chi_square_results.append({
        "champion_id": champion_id,
        "appearances": int(row["appearances"]),
        "wins": int(row["wins"]),
        "losses": int(row["losses"]),
        "win_rate": row["win_rate"],
        "chi_square": statistic,
        "p_value": p_value
    })

chi_square_results = pd.DataFrame(chi_square_results).sort_values("p_value")
bonferroni_alpha = alpha / len(chi_square_results)
chi_square_results["significant_at_5_percent"] = chi_square_results["p_value"] < alpha
chi_square_results["significant_after_bonferroni"] = (
    chi_square_results["p_value"] < bonferroni_alpha
)

# A pooled test is included for completeness, but is balanced by construction:
# every match contributes five winning and five losing champion appearances.
pooled_observed = [champion_summary["wins"].sum(), champion_summary["losses"].sum()]
pooled_expected = [sum(pooled_observed) / 2, sum(pooled_observed) / 2]
pooled_statistic, pooled_p_value = stats.chisquare(
    pooled_observed,
    f_exp=pooled_expected
)

print(f"Pooled chi-square statistic: {pooled_statistic:.2f}")
print(f"Pooled p-value: {pooled_p_value:.6g}")
print(f"Champions significant at 5%: {chi_square_results['significant_at_5_percent'].sum()}")
print(f"Champions significant after Bonferroni correction: {chi_square_results['significant_after_bonferroni'].sum()}")

if chi_square_results["significant_after_bonferroni"].any():
    print("Conclusion: Reject the null hypothesis for the affected champions; their win rates differ significantly from 50%.")
else:
    print("Conclusion: Fail to reject the null hypothesis; no champion differs significantly from 50% after correction.")

chi_square_results.head(10)

Pooled chi-square statistic: 0.00
Pooled p-value: 1
Champions significant at 5%: 53
Champions significant after Bonferroni correction: 23
Conclusion: Reject the null hypothesis for the affected champions; their win rates differ significantly from 50%.


,champion_id,appearances,wins,losses,win_rate,chi_square,p_value,significant_at_5_percent,significant_after_bonferroni
137,516,4708,1932,2776,41.036534,151.303314,8.996930e-35,True,True
39,40,8617,4784,3833,55.518162,104.955437,1.249138e-24,True,True
59,64,9116,4186,4930,45.919263,60.721369,6.575250e-15,True,True
36,37,5384,2920,2464,54.234770,38.621100,5.145961e-10,True,True
129,412,12875,6085,6790,47.262136,38.603883,5.191552e-10,True,True
12,13,1021,416,605,40.744368,34.986288,3.320353e-09,True,True
42,43,3114,1408,1706,45.215157,28.517662,9.284764e-08,True,True
28,29,8759,4623,4136,52.779998,27.077178,1.954923e-07,True,True
121,236,8238,3884,4354,47.147366,26.814761,2.239204e-07,True,True
71,81,5281,2458,2823,46.544215,25.227230,5.095764e-07,True,True


This chi-square test compares the win and loss rate of each individual champion to the expected 50/50 split. at a 5% significance level, 53 champions significantly differ from the given split. After utilising the Bonferroni correction, 23 champions remained significant.

Due to the nature of the game, with 5 guaranteed winners and 5 guaranteed losers each game, this is a very significant imbalance in the game system. Due to this, the null hypothesis will be rejected.

---

# Hypothesis 3: Using Two-Sample Independent T-Test

to determine whether popular champions suffer lower win rates compared to less popular champions.

* Alternative Hypothesis: Highly popular champions have a significantly lower mean win rate compared to moderately popular champions

* Null Hypothesis: There is no significant difference in mean win rates across different popularity tiers.

Unlike the first hypothesis, Only the top 10% of champions will be utilised. As well as this, rather than involving the least popular characters, as was done in the first hypothesis, instead, champions seated between the lower and upper quartile will be used as the "moderately popular characters".

In [33]:
win_rate = [f"{column}_winrate" for column in champion_columns]
champion_win_rate_data = pd.concat(
    [
        df[[champion_column, win_rate_column]].rename(
            columns={champion_column: "champion_id", win_rate_column: "win_rate"}
        )
        for champion_column, win_rate_column in zip(champion_columns, win_rate)
    ],
    ignore_index=True
)

champion_columns = [
    f"{team}_champ{champion_number}id"
    for team in ("t1", "t2")
    for champion_number in range(1, 6)
]

champion_records = []
for champion_column in champion_columns:
    team = 1 if champion_column.startswith("t1_") else 2
    champion_records.append(
        pd.DataFrame({
            "champion_id": df[champion_column],
            "win": (df["winner"] == team).astype(int)
        })
    )

champion_results = pd.concat(champion_records, ignore_index=True)
champion_summary = champion_results.groupby("champion_id")["win"].agg(
    appearances="count",
    wins="sum"
)
champion_summary["losses"] = champion_summary["appearances"] - champion_summary["wins"]
champion_summary["win_rate"] = champion_summary["wins"] / champion_summary["appearances"] * 100

champion_picks = df[champion_columns].stack().value_counts()
champion_pick_rates = (champion_picks / champion_picks.sum() * 100).rename("pick_rate")
champion_pick_rates.index.name = "champion_id"
champion_pick_rates = champion_pick_rates.sort_values(ascending=False)

champion_summary = champion_summary.join(champion_pick_rates, on="champion_id")
champion_summary

,appearances,wins,losses,win_rate,pick_rate
champion_id,,,,,
1,3259,1637,1622,50.230132,0.638356
2,1559,746,813,47.851187,0.305369
3,2697,1262,1435,46.792733,0.528275
4,3550,1829,1721,51.521127,0.695356
5,3226,1667,1559,51.673900,0.631892
...,...,...,...,...,...
429,3536,1694,1842,47.907240,0.692614
432,3693,1767,1926,47.847279,0.723366
497,6772,3396,3376,50.147667,1.326465


I first combined code from hypothesis 2 and 3 in order to generate champion_summary, which holds the appearances, wins, losses, win_rate and pick_rate of each character. Following this, I identified the champions in the top 10%, as well as those that sat in the "middle 50%" (within the quartiles)

In [ ]:
# Top 10% cutoff
top10pct_cutoff = champion_summary["pick_rate"].quantile(0.90)
# Identifying champions above the cutoff
top10pct = champion_summary[champion_summary["pick_rate"] >= top10pct_cutoff]
# Bottom 25% cutoff
midpct_cutoff_lower = champion_summary["pick_rate"].quantile(0.25)
# top 25% cutoff
midpct_cutoff_upper = champion_summary["pick_rate"].quantile(0.75)
# Cutting off champs lower than 25%, higher than 75%
midpct = champion_summary[(champion_summary["pick_rate"] >= midpct_cutoff_lower) & (champion_summary["pick_rate"] <= midpct_cutoff_upper)]

top10pct_mean = top10pct["pick_rate"].mean()
midpct_mean = midpct["pick_rate"].mean()
# Descriptive analysis of the top 10% and middle 50% pick rates
print(f"Top 10 percent Mean pick Rate: {top10pct_mean:.2f}%")
print(f"Middle 50 percent Mean pick Rate: {midpct_mean:.2f}%\n")

Top 10 percent Mean pick Rate: 1.77%
Middle 50 percent Mean pick Rate: 0.61%



From the above code, it is clear that the top 10% has a higher average pick rate than the middle 50%. Due to this, we are able to now conduct the statistical analysis.

In [71]:
test_top10pct = top10pct["pick_rate"].values
test_midpct = midpct["pick_rate"].values

# Conducting the t-test.
t_stat, p_value = stats.ttest_ind(
    test_top10pct, test_midpct, equal_var=False
)

print("--- T-TEST OUTPUT ---")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Interpreting the results
alpha = 0.05
if p_value < alpha:
    print("Reject the null hypothesis: There is a significant difference in pick rates between the top 10% and middle 50% champions.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in pick rates between the top 10% and middle 50% champions.")

--- T-TEST OUTPUT ---
T-Statistic: 11.3994
P-value: 0.0000
Reject the null hypothesis: There is a significant difference in pick rates between the top 10% and middle 50% champions.


From the conducted test, it is clear that the p_value is much smaller than alpha. Due to this, there is a very significant statistical strength to reject the null hypothesis. Howver, the T - Statistic being so positive actually suggests the alternative hypothesis to be false too. Instead, higher popularity champions seem to carry with them a higher win-rate. This may be explained by characters that happen to perform better being more popular to the playerbase, especially at a higher level of play, which this dataset is focused on, where matches are taken much more seriously.

---

# Hypothesis 4: Using Binomial Test for Proportions

To determine if an unfair advantage is given to a team.

* Alternative Hypothesis: The win rate of matches played on the Blue Team (Team1) significantly deviates from a 50% distribution, proving the existence of an advantage.

* Null Hypothesis: The Teams are perfectly fair; Blue Team and Red Team win exactly 50% of matches each.

In League of Legends, the map is almost symmetrical, but is not. There are minor differences that may possibly give an advantage to one team over the other. Due to this, I decided this hypothesis is worth investigating, as there is a given reason why there might not be a 50/50 split in wins for each team, other than the other factors already accounted for.

In [119]:
total_matches = len(df)
total_matches

blue_wins = int((df["winner"] == 1).sum())
red_wins = int((df["winner"] == 2).sum())

blue_win_rate = blue_wins / total_matches
red_win_rate = red_wins / total_matches

print(f"Total Matches Extracted: {total_matches}")
print(f"Blue Team Wins (1): {blue_wins} ({blue_win_rate * 100:.2f}%)")
print(f"Red Team Wins (2): {red_wins} ({red_win_rate * 100:.2f}%)\n")

result = stats.binomtest(k=blue_wins, n=total_matches, p=0.5)

print("--- STATISTICAL OUTPUT ---")
print(f"Estimated Blue Proportion (p-hat): {result.statistic:.4f}")
print(f"P-value: {result.pvalue:.4f}")

alpha = 0.05
if result.pvalue < alpha:
    print("Reject the null hypothesis: The win rate for the blue team is significantly different from 50%.")
else:
    print("Fail to reject the null hypothesis: The win rate for the blue team is not significantly different from 50%.")

Total Matches Extracted: 51053
Blue Team Wins (1): 25857 (50.65%)
Red Team Wins (2): 25196 (49.35%)

--- STATISTICAL OUTPUT ---
Estimated Blue Proportion (p-hat): 0.5065
P-value: 0.0035
Reject the null hypothesis: The win rate for the blue team is significantly different from 50%.


According to the test completed above, the p_value suggests that we are to reject the null hypothesis, with it being slightly smaller than the alpha. Because of this, we can conclude that there is a measurable advantage towards the blue team.

---

# Final Conclusions and Insights

After completing statistical analysis related to the above hypotheses, some conclusions can be drawn.

* Low pick rate characters have equal volatility to those in high pick rate.

* There is an inbalance in winrates across the many champions.

* Higher popularity characters also hold a higher win rate.

* The blue team has an advantage over the red team.